#### PDF Password Removal

In [8]:
import random
import string
import pikepdf
from pikepdf import Name, Dictionary

#### Generate a password protected file locally

In [9]:
def create_random_protected_pdf(output_path: str, password: str) -> None:
    # 1. Initialize a new PDF and add a standard Letter-sized blank page
    pdf = pikepdf.new()
    page = pdf.add_blank_page(page_size=(612, 792))

    # 2. Register a standard built-in PDF font (Helvetica) so the text renders
    font_dict = Dictionary(
        Type=Name.Font,
        Subtype=Name.Type1,
        BaseFont=Name.Helvetica
    )
    page.Resources = Dictionary(
        Font=Dictionary(F1=pdf.make_indirect(font_dict))
    )

    # 3. Generate random lines of data
    random_lines = [
        "CONFIDENTIAL TEST DOCUMENT",
        "----------------------------------------"
    ]
    for i in range(1, 16):
        rand_token = ''.join(random.choices(string.ascii_uppercase + string.digits, k=12))
        rand_val = round(random.uniform(100.0, 9999.99), 2)
        random_lines.append(f"Record #{i:02d} | ID: {rand_token} | Value: ${rand_val}")

    # 4. Build the raw PDF content stream to draw the text on the page
    # BT = Begin Text, /F1 12 Tf = Font F1 at 12pt, Td = Move coordinates, Tj = Show text, ET = End Text
    content_commands = ["BT", "/F1 12 Tf", "50 720 Td"]
    for line in random_lines:
        # Escape parentheses if any exist in the text
        safe_line = line.replace("(", r"\(").replace(")", r"\)")
        content_commands.append(f"({safe_line}) Tj")
        content_commands.append("0 -22 Td")  # Move down 22 points for the next line
    content_commands.append("ET")

    # Attach the stream to the page
    stream_bytes = "\n".join(content_commands).encode("ascii")
    page.Contents = pdf.make_stream(stream_bytes)

    # 5. Save the PDF with AES-256 encryption (R=6)
    encryption_settings = pikepdf.Encryption(
        user=password,   # Password required to open and view the PDF
        owner=password,  # Password required to change permissions
        R=6              # R=6 specifies AES-256 encryption
    )
    
    pdf.save(output_path, encryption=encryption_settings)
    print(f"Created '{output_path}' with random data.")
    print(f"Password to open: {password}")

In [5]:
# Create a blank PDF with 1 page
pdf = pikepdf.new()
pdf.add_blank_page(page_size=(612, 792))  # Standard Letter size

# Save it with an owner and user password
password = "secret123"
pdf.save(
    "protected.pdf",
    encryption=pikepdf.Encryption(owner=password, user=password, R=6)  # R=6 uses AES-256
)

print("Created 'protected.pdf' locked with password: secret123")

Created 'protected.pdf' locked with password: secret123


#### Code for removing the pdf password and saving the file locally to disk

In [6]:
def remove_pdf_password(input_path: str, output_path: str, password: str) -> None:
    try:
        # Open the password-protected PDF
        with pikepdf.open(input_path, password=password) as pdf:
            # Saving the file writes an unencrypted version by default
            pdf.save(output_path)
        print(f"Success! Unprotected PDF saved to: {output_path}")
        
    except pikepdf.PasswordError:
        print("Error: The password provided is incorrect.")
    except FileNotFoundError:
        print(f"Error: The file '{input_path}' was not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [11]:
if __name__ == "__main__": 
    create_random_protected_pdf("protected.pdf", "secret123")
    output_pdf = "unprotected.pdf"
    pdf_password = "secret123"
    remove_pdf_password(input_pdf, output_pdf, pdf_password)

Created 'protected.pdf' with random data.
Password to open: secret123
Success! Unprotected PDF saved to: unprotected.pdf
